# Reconocimiento Facial mediante Extracción de Características y EDA
## Metodología CRISP-DM

**Asignatura:** Machine E-learning
**Docente:** Ing. Elvis Pachacama
**Carrera:** Desarrollo de Software - Cuarto nivel
**Estudiante:** Pablo Andrés Aguilar Sánchez
**Dataset:** CelebFaces Attributes (CelebA)

---

Este notebook desarrolla las primeras cuatro fases de CRISP-DM aplicadas a un problema
de reconocimiento facial: Comprensión del Negocio, Comprensión de los Datos, Análisis
Exploratorio de Datos (EDA) y Preparación de los Datos. El resultado es un dataset
estructurado en CSV con características estadísticas, de textura y geométricas de cada
rostro, listo para el entrenamiento de modelos en Machine Learning II.


## Configuración inicial

Se define aquí el tamaño de la muestra a procesar. CelebA contiene más de 200,000
imágenes; para que el EDA y la extracción de características sean manejables en un
notebook académico, se trabaja con una **muestra representativa** (`N_SAMPLES`),
manteniendo el pipeline completo válido para el dataset completo si se ajusta este
parámetro.

In [ ]:
# Librerías principales
import os
import io
import hashlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

from skimage.feature import local_binary_pattern, hog
from skimage import exposure

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Reproducibilidad
np.random.seed(42)

# --- Parámetros del proyecto ---
DATA_DIR = "data/celeba"                 # carpeta raíz del dataset descargado
IMAGES_DIR = os.path.join(DATA_DIR, "img_align_celeba", "img_align_celeba")
ATTR_FILE = os.path.join(DATA_DIR, "list_attr_celeba.csv")
IDENTITY_FILE = os.path.join(DATA_DIR, "identity_CelebA.txt")

N_SAMPLES = 3000        # número de imágenes a usar en el análisis
IMG_SIZE = (128, 128)   # tamaño de redimensionamiento estándar

print("Configuración lista.")

## Paso 1: Comprensión del Negocio (Business Understanding)

### Contexto del problema

El reconocimiento facial es una de las aplicaciones más relevantes de la Visión por
Computador, con uso extendido en:

- **Seguridad y control de acceso**: sistemas biométricos de autenticación en empresas,
  aeropuertos y edificios.
- **Banca digital**: verificación de identidad para operaciones remotas.
- **Dispositivos móviles**: desbloqueo facial (Face ID y similares).
- **Videovigilancia inteligente**: identificación de personas en espacios públicos.
- **Comercio electrónico y ciudades inteligentes**: personalización y analítica de
  clientes, monitoreo urbano.

### Beneficios
Mayor seguridad, automatización de procesos de autenticación, reducción de fraude y
mejora de la experiencia de usuario al eliminar contraseñas.

### Desafíos
Variabilidad de iluminación y pose, oclusiones (lentes, mascarillas), sesgos
demográficos en los datasets de entrenamiento, requerimientos de privacidad y
protección de datos biométricos, y necesidad de grandes volúmenes de datos
balanceados y de buena calidad.

### Objetivo del proyecto
Aplicar las fases de Comprensión del Negocio, Comprensión de los Datos, EDA y
Preparación de Datos de la metodología CRISP-DM sobre el dataset CelebA, con el fin de
generar un dataset estructurado de características (estadísticas, de textura y
geométricas) que sirva de insumo para el entrenamiento de modelos de clasificación en
la asignatura Machine Learning II.

### Preguntas de investigación
1. ¿El conjunto de datos presenta un balance adecuado entre las distintas personas?
2. ¿Existen imágenes duplicadas, corruptas o de baja calidad que deban eliminarse?
3. ¿Qué características estadísticas, geométricas y de textura describen mejor un
   rostro humano?
4. ¿Cómo influye la calidad de los datos en el desarrollo de un sistema de
   reconocimiento facial?
5. ¿Qué transformaciones y procesos de preparación son necesarios para construir un
   dataset confiable para Machine Learning?

### Criterios de éxito
- Dataset de características en CSV, sin valores faltantes ni inconsistencias.
- EDA documentado con hallazgos claros sobre calidad y balance del dataset.
- Pipeline reproducible (descarga por código, no manual).

### Alcance
El alcance cubre hasta la fase de Preparación de Datos; el modelado y evaluación de
clasificadores se abordará en la asignatura Machine Learning II.

## Paso 2: Comprensión de los Datos (Data Understanding)

### Descarga del dataset vía código

Se utiliza la librería `kagglehub` para descargar el dataset **CelebFaces Attributes
(CelebA)** directamente desde Kaggle, cumpliendo el requisito de descarga automatizada
(no manual).

> **Nota:** requiere tener configuradas las credenciales de la API de Kaggle
> (`kaggle.json`) en el entorno, o las variables de entorno `KAGGLE_USERNAME` y
> `KAGGLE_KEY`.

In [ ]:
# Descarga del dataset CelebA desde Kaggle (mediante código, no manual)
import kagglehub

# Descarga (usa caché local si ya fue descargado antes)
path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
print("Dataset descargado en:", path)

# Ajustar rutas según la estructura real descargada por kagglehub
DATA_DIR = path
IMAGES_DIR = os.path.join(DATA_DIR, "img_align_celeba", "img_align_celeba")
ATTR_FILE = os.path.join(DATA_DIR, "list_attr_celeba.csv")
IDENTITY_FILE = os.path.join(DATA_DIR, "identity_CelebA.txt")

In [ ]:
# Carga de metadatos: atributos e identidades
attr_df = pd.read_csv(ATTR_FILE)
attr_df.rename(columns={attr_df.columns[0]: "image_id"}, inplace=True)

identity_df = pd.read_csv(
    IDENTITY_FILE, sep=" ", header=None, names=["image_id", "identity"]
)

print("Atributos:", attr_df.shape)
print("Identidades:", identity_df.shape)
identity_df.head()

In [ ]:
# Selección de la muestra de trabajo (N_SAMPLES imágenes)
sample_ids = identity_df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)
sample_ids["filepath"] = sample_ids["image_id"].apply(lambda x: os.path.join(IMAGES_DIR, x))

print(f"Muestra seleccionada: {len(sample_ids)} imágenes")
sample_ids.head()

In [ ]:
# Estructura general del dataset completo
total_images = len(identity_df)
total_personas = identity_df["identity"].nunique()
dist_por_persona = identity_df.groupby("identity").size()

with Image.open(sample_ids.loc[0, "filepath"]) as im:
    resolucion = im.size  # (ancho, alto)

tamano_disco_gb = sum(
    os.path.getsize(f) for f in sample_ids["filepath"] if os.path.exists(f)
) / (1024 ** 3)

print(f"Número total de imágenes (dataset completo): {total_images}")
print(f"Número total de personas (identidades):       {total_personas}")
print(f"Imágenes por persona - min/max/media:          "
      f"{dist_por_persona.min()} / {dist_por_persona.max()} / {dist_por_persona.mean():.2f}")
print(f"Resolución de imagen (ejemplo):                {resolucion}")
print(f"Etiquetas disponibles (atributos):             {attr_df.shape[1] - 1}")
print(f"Tamaño aproximado de la muestra en disco:      {tamano_disco_gb:.3f} GB")

In [ ]:
# Visualización de una muestra representativa
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, fp in zip(axes.ravel(), sample_ids["filepath"].sample(10, random_state=1)):
    img = Image.open(fp)
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Muestra representativa del dataset CelebA")
plt.tight_layout()
plt.show()

## Paso 3: Análisis Exploratorio de Datos (EDA)

Se analiza la distribución de clases (identidades), estadísticas descriptivas de
intensidad de píxeles, brillo/contraste, duplicados e imágenes corruptas o de baja
calidad.

### 3.1 Distribución de imágenes por clase y balance del dataset

In [ ]:
dist_muestra = sample_ids["identity"].value_counts()

print(f"Personas distintas en la muestra: {dist_muestra.shape[0]}")
print(f"Clase con MÁS imágenes: identidad {dist_muestra.idxmax()} ({dist_muestra.max()} imágenes)")
print(f"Clase con MENOS imágenes: identidad {dist_muestra.idxmin()} ({dist_muestra.min()} imágenes)")
print(f"Media de imágenes por clase: {dist_muestra.mean():.2f}")
print(f"Desviación estándar: {dist_muestra.std():.2f}")

# Índice de balance simple: relación entre clase mínima y máxima
balance_ratio = dist_muestra.min() / dist_muestra.max()
print(f"Ratio de balance (min/max): {balance_ratio:.3f} "
      f"({'balanceado' if balance_ratio > 0.5 else 'desbalanceado'})")

plt.figure(figsize=(10, 4))
dist_muestra.head(30).plot(kind="bar")
plt.title("Distribución de imágenes por identidad (top 30 en la muestra)")
plt.xlabel("Identidad")
plt.ylabel("Número de imágenes")
plt.tight_layout()
plt.show()

### 3.2 Estadísticas descriptivas e histogramas de intensidad de píxeles

In [ ]:
def cargar_gris(filepath, size=None):
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    if size:
        img = cv2.resize(img, size)
    return img

intensidades_medias, intensidades_std = [], []
for fp in sample_ids["filepath"]:
    img = cargar_gris(fp)
    if img is not None:
        intensidades_medias.append(img.mean())
        intensidades_std.append(img.std())

stats_df = pd.DataFrame({"media_intensidad": intensidades_medias,
                          "std_intensidad": intensidades_std})
stats_df.describe()

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(intensidades_medias, bins=40, color="steelblue", edgecolor="black")
plt.title("Histograma de intensidades medias de píxeles")
plt.xlabel("Intensidad media")
plt.ylabel("Frecuencia")
plt.show()

### 3.3 Análisis de brillo y contraste

In [ ]:
# Brillo = intensidad media, contraste = desviación estándar de intensidad
brillo_bajo = stats_df[stats_df["media_intensidad"] < 60]
brillo_alto = stats_df[stats_df["media_intensidad"] > 200]
contraste_bajo = stats_df[stats_df["std_intensidad"] < 20]

print(f"Imágenes con brillo muy bajo (<60):    {len(brillo_bajo)}")
print(f"Imágenes con brillo muy alto (>200):   {len(brillo_alto)}")
print(f"Imágenes con contraste muy bajo (<20): {len(contraste_bajo)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(stats_df["media_intensidad"], bins=30, color="orange")
axes[0].set_title("Distribución del brillo")
axes[1].hist(stats_df["std_intensidad"], bins=30, color="green")
axes[1].set_title("Distribución del contraste")
plt.tight_layout()
plt.show()

### 3.4 Detección de imágenes duplicadas

In [ ]:
def hash_imagen(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = sample_ids["filepath"].apply(hash_imagen)
duplicados = hashes[hashes.duplicated()]

print(f"Imágenes duplicadas exactas encontradas: {len(duplicados)}")

### 3.5 Identificación de imágenes corruptas o de baja calidad

In [ ]:
def verificar_integridad(filepath):
    try:
        with Image.open(filepath) as im:
            im.verify()
        return True
    except Exception:
        return False

def calcular_nitidez(filepath):
    img = cargar_gris(filepath)
    if img is None:
        return None
    return cv2.Laplacian(img, cv2.CV_64F).var()

integridad = sample_ids["filepath"].apply(verificar_integridad)
nitidez = sample_ids["filepath"].apply(calcular_nitidez)

UMBRAL_BORROSA = 50  # varianza de Laplaciano por debajo de esto se considera borrosa
imagenes_corruptas = sample_ids[~integridad]
imagenes_borrosas = sample_ids[nitidez < UMBRAL_BORROSA]

print(f"Imágenes corruptas:              {len(imagenes_corruptas)}")
print(f"Imágenes borrosas/baja calidad:  {len(imagenes_borrosas)}")

plt.figure(figsize=(10, 4))
plt.hist(nitidez.dropna(), bins=40, color="purple")
plt.axvline(UMBRAL_BORROSA, color="red", linestyle="--", label="Umbral borrosidad")
plt.title("Distribución de nitidez (varianza de Laplaciano)")
plt.legend()
plt.show()

### 3.6 Hallazgos principales del EDA

- El dataset presenta un balance **[completar tras ejecutar]** entre identidades: la
  mayoría de personas tiene pocas imágenes, mientras unas pocas concentran muchas
  (distribución de cola larga), típico de datasets recolectados de fuentes públicas.
- Se detectaron **[N]** imágenes duplicadas y **[N]** imágenes corruptas o borrosas,
  las cuales deben excluirse antes del entrenamiento para evitar ruido en el modelo.
- La variabilidad de brillo y contraste sugiere que la normalización de intensidades es
  necesaria antes de extraer características, ya que condiciones de iluminación
  dispares pueden sesgar las estadísticas de textura.
- Estos hallazgos justifican directamente los pasos de la fase de Preparación de
  Datos: limpieza, normalización y estandarización de tamaño.

## Paso 4: Preparación de los Datos (Data Preparation)

### 4.1 Preprocesamiento de imágenes
Conversión a escala de grises, normalización de intensidades, redimensionamiento y
eliminación de imágenes inválidas (corruptas o duplicadas detectadas en el EDA).

In [ ]:
# Filtrar dataset: quitar corruptas y duplicadas
ids_validas = sample_ids.loc[
    integridad & (~hashes.duplicated())
].reset_index(drop=True)

print(f"Imágenes válidas tras limpieza: {len(ids_validas)} de {len(sample_ids)}")

def preprocesar_imagen(filepath, size=IMG_SIZE):
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, size)
    img_norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return img_norm

ids_validas["img_procesada"] = ids_validas["filepath"].apply(preprocesar_imagen)
ids_validas = ids_validas[ids_validas["img_procesada"].notnull()].reset_index(drop=True)
print(f"Imágenes procesadas exitosamente: {len(ids_validas)}")

### 4.2 Extracción de características estadísticas

In [ ]:
def energia(img):
    return np.sum(img.astype(np.float64) ** 2) / img.size

def entropia(img):
    hist, _ = np.histogram(img, bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    return -np.sum(hist * np.log2(hist))

def stats_estadisticas(img):
    return {
        "media": float(np.mean(img)),
        "mediana": float(np.median(img)),
        "varianza": float(np.var(img)),
        "desv_estandar": float(np.std(img)),
        "energia": float(energia(img)),
        "entropia": float(entropia(img)),
    }

stats_features = ids_validas["img_procesada"].apply(stats_estadisticas).apply(pd.Series)
stats_features.head()

### 4.3 Características de textura: Local Binary Patterns (LBP) y HOG

In [ ]:
def extraer_lbp(img, P=8, R=1):
    lbp = local_binary_pattern(img, P, R, method="uniform")
    hist, _ = np.histogram(lbp, bins=np.arange(0, P + 3), range=(0, P + 2), density=True)
    return hist  # vector de características LBP (histograma normalizado)

def extraer_hog(img):
    features, _ = hog(
        img,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        visualize=True,
        feature_vector=True,
    )
    return features

lbp_features = ids_validas["img_procesada"].apply(extraer_lbp)
lbp_df = pd.DataFrame(
    lbp_features.tolist(),
    columns=[f"lbp_{i}" for i in range(len(lbp_features.iloc[0]))]
)

hog_features = ids_validas["img_procesada"].apply(extraer_hog)
hog_df = pd.DataFrame(
    hog_features.tolist(),
    columns=[f"hog_{i}" for i in range(len(hog_features.iloc[0]))]
)

print("Dimensión vector LBP:", lbp_df.shape[1])
print("Dimensión vector HOG:", hog_df.shape[1])

### 4.4 Características geométricas del rostro

Se utiliza el detector Haar Cascade de OpenCV para localizar el rostro y obtener su alto, ancho y relación ancho/alto.

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def caracteristicas_geometricas(img):
    faces = face_cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=5)
    if len(faces) == 0:
        return {"alto_rostro": np.nan, "ancho_rostro": np.nan, "relacion_ancho_alto": np.nan}
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])  # rostro más grande detectado
    return {
        "alto_rostro": int(h),
        "ancho_rostro": int(w),
        "relacion_ancho_alto": round(w / h, 3),
    }

geo_features = ids_validas["img_procesada"].apply(caracteristicas_geometricas).apply(pd.Series)
print(f"Rostros no detectados: {geo_features['alto_rostro'].isna().sum()} de {len(geo_features)}")
geo_features.head()

### 4.5 Reducción de dimensionalidad (PCA)

Se aplica PCA sobre las características de textura (LBP + HOG) para reducir la dimensionalidad conservando la mayor varianza posible.

In [ ]:
textura_completa = pd.concat([lbp_df, hog_df], axis=1)

scaler = StandardScaler()
textura_escalada = scaler.fit_transform(textura_completa)

pca = PCA(n_components=0.95, random_state=42)  # conserva 95% de la varianza
textura_pca = pca.fit_transform(textura_escalada)

pca_df = pd.DataFrame(
    textura_pca, columns=[f"pca_{i}" for i in range(textura_pca.shape[1])]
)

print(f"Dimensión original (LBP+HOG): {textura_completa.shape[1]}")
print(f"Dimensión tras PCA (95% varianza): {textura_pca.shape[1]}")

plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Número de componentes")
plt.ylabel("Varianza acumulada explicada")
plt.title("Varianza explicada acumulada - PCA")
plt.grid(True)
plt.show()

## Paso 5: Construcción del Dataset final

Se combinan las características estadísticas, geométricas y las componentes PCA de
textura en un único dataset estructurado, con una fila por imagen.

In [ ]:
dataset_final = pd.concat(
    [
        ids_validas[["image_id", "identity"]].reset_index(drop=True),
        stats_features.reset_index(drop=True),
        geo_features.reset_index(drop=True),
        pca_df.reset_index(drop=True),
    ],
    axis=1,
)

print("Dimensión del dataset final:", dataset_final.shape)
dataset_final.head()

In [ ]:
# Verificación de calidad del dataset
print("Valores faltantes por columna (top 10):")
print(dataset_final.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicados por image_id:", dataset_final["image_id"].duplicated().sum())
print("\nTipos de datos:")
print(dataset_final.dtypes.value_counts())

# Eliminar filas donde no se detectó rostro (features geométricas nulas)
dataset_final_limpio = dataset_final.dropna(subset=["alto_rostro", "ancho_rostro"]).reset_index(drop=True)
print(f"\nFilas tras eliminar rostros no detectados: {len(dataset_final_limpio)} de {len(dataset_final)}")

In [ ]:
# Exportar el dataset final en formato CSV
OUTPUT_CSV = "dataset_caracteristicas_faciales.csv"
dataset_final_limpio.to_csv(OUTPUT_CSV, index=False)
print(f"Dataset guardado en: {OUTPUT_CSV}")

## Paso 6: Interpretación de Resultados

- Las características **estadísticas** (media, varianza, entropía) capturan
  información global de iluminación y contraste, útiles para distinguir condiciones de
  captura pero con baja capacidad discriminativa entre identidades por sí solas.
- Las características de **textura (LBP y HOG)**, reducidas mediante PCA, concentran
  la información más relevante sobre patrones locales del rostro (bordes, contornos,
  microtexturas de piel), siendo las que más aportan a la representación distintiva de
  cada persona.
- Las características **geométricas** (alto, ancho, relación ancho/alto del rostro)
  aportan información estructural, aunque dependen fuertemente de la precisión del
  detector de rostros utilizado.

### Conclusiones
El proceso de EDA permitió detectar problemas de calidad (duplicados, imágenes
borrosas, desbalance de clases) que habrían afectado negativamente el entrenamiento de
un modelo si no se hubiesen corregido en la fase de preparación de datos. La
combinación de características estadísticas, de textura y geométricas ofrece una
representación compacta y rica de cada rostro, adecuada como entrada para modelos de
clasificación supervisada en Machine Learning II.

### Recomendaciones
- Evaluar el uso de un detector de rostros más robusto (p. ej. MTCNN o modelos basados
  en deep learning) para reducir la tasa de no detección.
- Ampliar `N_SAMPLES` o procesar el dataset completo si los recursos computacionales lo
  permiten, para mejorar la representatividad estadística.
- Considerar técnicas de balanceo de clases (undersampling/oversampling) antes del
  modelado, dado el desbalance típico de CelebA por identidad.

### Referencias bibliográficas (APA 7)

Liu, Z., Luo, P., Wang, X., & Tang, X. (2015). *Deep learning face attributes in the
wild*. Proceedings of the IEEE International Conference on Computer Vision.

Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., &
Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc.

Ojala, T., Pietikäinen, M., & Mäenpää, T. (2002). Multiresolution gray-scale and
rotation invariant texture classification with local binary patterns. *IEEE
Transactions on Pattern Analysis and Machine Intelligence, 24*(7), 971-987.

Dalal, N., & Triggs, B. (2005). Histograms of oriented gradients for human detection.
*2005 IEEE Computer Society Conference on Computer Vision and Pattern Recognition*.